# R-GCN y CompGCN — Ediets

Solo estos dos modelos y solo el grafo de Ediets (68 cláusulas, 300 enunciados). Sus partes **no están verificadas a mano**: en el
paso 5, «la parte correcta» es la que puso la extracción. Mismos pasos y mismas pruebas que
[`transe_transh_transr/Ediets.ipynb`](../transe_transh_transr/Ediets.ipynb), con otros modelos:

1. el grafo como tripletas `(h, r, t)`;
2. qué hace cada modelo;
3. entrenarlos y ver qué han aprendido;
4. ¿adivinan aristas que no han visto?;
5. la pregunta que nos importa: ¿adivinan la parte de un enunciado?, y por qué no.

In [ ]:
import collections
import sys
import time
from pathlib import Path

import numpy as np
import torch
from IPython.display import Markdown, display

# link_prediction.py está en notebooks/, una carpeta por encima de este cuaderno.
for folder in (Path.cwd(), Path.cwd().parent, Path.cwd() / "notebooks"):
    if (folder / "link_prediction.py").exists():
        sys.path.insert(0, str(folder))
        break
import link_prediction as lp

# Configuración de cada modelo: la que eligió el ajuste de link_prediction.ipynb (sección 3), hecho
# en SteelVault. Aquí se usa tal cual, sin ajustarla a este contrato.
#   loss           "1vsall": cada tripleta verdadera contra todas las entidades a la vez
#                  "margin": contra 32 tripletas falsas hechas al azar (pérdida de margen)
#   lr             tasa de aprendizaje: cuánto se mueven los parámetros en cada corrección
#   epochs         cuántas veces se recorren todas las tripletas
#   embedding_dim  números por entidad, en el vector de partida y en el que sale de cada capa
#   batch_size     tripletas por corrección
#   negatives      tripletas falsas por cada verdadera (solo con "margin")
# Los dos pasan mensajes en 2 capas (lo fija lp.train_kge) y puntúan con DistMult (lo que PyKEEN
# usa por defecto). El optimizador es Adam.
CONFIG = {
    "RGCN": {"loss": "1vsall", "lr": 0.03, "epochs": 300, "embedding_dim": 128, "batch_size": 256, "negatives": 32},
    "CompGCN": {"loss": "1vsall", "lr": 0.1, "epochs": 300, "embedding_dim": 32, "batch_size": 256, "negatives": 32},
}
MODELS = tuple(CONFIG)
# Veces que se repite el paso 5 con repartos distintos. 1, como link_prediction.ipynb hizo con
# este contrato; con 3, el paso 5 tarda el triple.
REPEATS = 1

contract_name = "Ediets"
contract = lp.load_contract(contract_name)
e2i, r2i = lp.entity_ids(contract)
i2e = {i: e for e, i in e2i.items()}
print("dispositivo:", lp.DEVICE, "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")

In [ ]:
ALL_TYPES = ("is_part_of", "assigns_obligation_to", "grants_right_to", "defines", "uses", "references", "depends_on", "supersedes", "modifies")
print(f"contrato {contract_name}")
print(f"{len(contract.triples)} tripletas, {len(e2i)} entidades, {len(r2i)} tipos de arista presentes")
print("tipos de arista que no aparecen:", ", ".join(r for r in ALL_TYPES if r not in r2i) or "ninguno")
lp.table([{"tipo de arista": r, "tripletas": n} for r, n in collections.Counter(r for _, r, _ in contract.triples).most_common()])

Una tripleta de cada tipo de arista, con el texto de cada nodo para poder leerlas:

In [ ]:
def label(node):
    if node in contract.parties:
        return f"{node} ({contract.parties[node]['name']})"
    if node in contract.clauses:
        c = contract.clauses[node]
        return f"{node} (§{c['ref']} {c.get('heading') or ''})".strip()
    if node in contract.statements:
        return f"{node} «{lp.statement_text(contract.statements[node])[:55]}…»"
    return node


examples = [next(t for t in contract.triples if t[1] == r) for r in sorted(r2i)]
lp.table([{"h": label(h), "r": r, "t": label(t)} for h, r, t in examples])

## 2. Qué hace cada modelo

TransE y los de [`rotate_distmult_complex`](../rotate_distmult_complex/Ediets.ipynb) guardan un
vector por entidad y puntúan una tripleta mirando solo los vectores de sus dos extremos. R-GCN y
CompGCN añaden un paso antes: el vector de cada entidad se **construye con los de sus vecinos**.
Cada entidad parte de un vector propio y, en cada capa, recibe un mensaje de cada vecino, por las
aristas en los dos sentidos. Con dos capas le llega lo que hay a uno y dos saltos. Después se
puntúa con los vectores construidos, como en DistMult: `⟨eₕ, r, eₜ⟩`.

| Modelo | El mensaje de un vecino es… |
|---|---|
| **R-GCN** | su vector multiplicado por una **matriz propia del tipo de arista** y de su sentido (cada matriz es una mezcla de unas pocas matrices base) |
| **CompGCN** | su vector **combinado con el vector del tipo de arista** (aquí, multiplicados número a número) y luego por una matriz según el sentido. Los vectores de los tipos de arista también se actualizan en cada capa |

Lo que distingue a esta familia: el vector de un nodo no es un punto libre que el entrenamiento
coloca donde quiere, sino un resumen de su vecindad. Dos enunciados con vecinos parecidos acaban
con vectores parecidos.

**El problema del muchos-a-uno.** Cuántas aristas de parte apuntan a cada parte en este contrato:

In [ ]:
counts = collections.Counter((r, t) for _, r, t in contract.triples if r in lp.PARTY_RELATIONS)
lp.table([{"tipo de arista": r, "parte": label(p), "aristas": n} for (r, p), n in sorted(counts.items())])

Aquí las aristas de parte también son **caminos de mensajes**: cada parte recibe el mensaje de
todos los enunciados que la apuntan, y su vector se construye en buena parte con la media de todos
ellos. Lo mismo la otra parte. Si los enunciados de las dos partes se parecen —obligaciones en las
mismas cláusulas, con las mismas condiciones—, los vectores de las dos partes también se parecerán,
y distinguirlas costará.

## 3. Entrenar y ver qué han aprendido

**Entrenar** es lo mismo que en TransE: para cada tripleta conocida se le pregunta al modelo
`(h, r, ?)`, se puntúan todas las entidades como posible destino y se corrige para que la
verdadera quede arriba. Lo mismo con el origen, `(?, r, t)`. Lo que se corrige aquí son los
vectores de partida, las matrices de cada capa y los vectores de cada tipo de arista. Se repite
trescientas veces sobre todas las tripletas (las *épocas*).

In [ ]:
models = {}
rows = []
for m in MODELS:
    start = time.time()
    models[m] = lp.train_kge(m, contract.triples, e2i, r2i, seed=0, **CONFIG[m])
    rows.append({"modelo": m, **CONFIG[m], "segundos": round(time.time() - start, 1)})
lp.table(rows, ["modelo", "epochs", "embedding_dim", "lr", "loss", "segundos"])

**Cómo se lee un modelo.**

Se le hace una pregunta: ¿a quién obliga una obligación concreta? La celda toma la primera
obligación del contrato que obliga a una sola parte. El modelo puntúa todas las entidades y las
ordena; la posición de la respuesta correcta es su **rango**: 1 si sale la primera.

Las cinco primeras de cada modelo. El modelo ha visto la respuesta al entrenar: esto mide
memoria, no deducción.

In [ ]:
example = next(sid for sid, r, _ in lp.party_queries(contract) if r == "assigns_obligation_to" and sid not in contract.twin and contract.statements[sid]["kind"] == "obligation")
answer = contract.party_edge(example)[1]
display(Markdown(f"Pregunta: `({example}, assigns_obligation_to, ?)`, «{lp.statement_text(contract.statements[example])[:100]}». Respuesta correcta: `{answer}` ({contract.parties[answer]['name']})."))


def top(model, h, r, k=5):
    scores = lp.kge_tail_scores(model, [(h, r)], e2i, r2i)[0]
    order = np.argsort(-scores)
    return [f"{i2e[i]} ({scores[i]:.2f})" for i in order[:k]]


lp.table([{"modelo": m, **{f"{k + 1}.º": x for k, x in enumerate(top(models[m], example, "assigns_obligation_to"))}} for m in MODELS])

Tiene el formato Entidad (puntuación). Solo importa el orden: cada modelo usa su propia escala.

## 4. ¿Adivinan aristas que no han visto?

Ahora la prueba de verdad: se esconde un 10% de las tripletas al azar, se entrena con el resto y
se pregunta por las escondidas. Para resumir todos esos rangos en un número se usan dos medidas:

- **MRR** (*rango recíproco medio*): la media de `1 / rango`. Si la respuesta sale 1.ª, 2.ª y
  4.ª en tres preguntas, el MRR es `(1 + 1/2 + 1/4) / 3 = 0,58`. Vale 1 si siempre sale la primera.
- **Hits@k**: la fracción de preguntas en que la respuesta sale entre las *k* primeras.

El rango es **filtrado**: si una pregunta tiene varias respuestas verdaderas conocidas (una
cláusula tiene muchos enunciados), las otras no cuentan como error. Como referencia, el azar:
ordenar todas las entidades sin mirar nada.

In [ ]:
rng = np.random.default_rng(0)
hidden_idx = set(rng.choice(len(contract.triples), size=len(contract.triples) // 10, replace=False).tolist())
train = [t for i, t in enumerate(contract.triples) if i not in hidden_idx]
hidden = [contract.triples[i] for i in sorted(hidden_idx)]
true_tails = collections.defaultdict(set)
for h, r, t in contract.triples:
    true_tails[(h, r)].add(e2i[t])


def filtered_ranks(model, triples):
    scores = lp.kge_tail_scores(model, [(h, r) for h, r, _ in triples], e2i, r2i)
    ranks = []
    for row, (h, r, t) in zip(scores, triples):
        row = row.copy()
        row[[i for i in true_tails[(h, r)] if i != e2i[t]]] = -np.inf
        ranks.append(1 + int(np.sum(row > row[e2i[t]])))
    return np.array(ranks)


n = len(e2i)
rows = [{"modelo": "al azar (esperado)", "MRR": float(np.mean([1 / k for k in range(1, n + 1)])), "Hits@1": 1 / n, "Hits@10": 10 / n}]
ranks_by_model = {}
for m in MODELS:
    ranks = filtered_ranks(lp.train_kge(m, train, e2i, r2i, seed=0, **CONFIG[m]), hidden)
    ranks_by_model[m] = ranks
    rows.append({"modelo": m, "MRR": float(np.mean(1 / ranks)), "Hits@1": float(np.mean(ranks == 1)), "Hits@10": float(np.mean(ranks <= 10))})
display(Markdown(f"{len(hidden)} tripletas escondidas de {len(contract.triples)}"))
lp.table(rows)

Como referencia, en el ajuste de [`link_prediction.ipynb`](../link_prediction.ipynb) —SteelVault,
otra muestra del 10% y dos semillas— el MRR fue 0,45 para R-GCN y 0,42 para CompGCN (TransE: 0,46).

Las escondidas son muchas para mirarlas una a una. Primero, el MRR de cada modelo según el tipo
de arista escondida:

In [ ]:
by_type = collections.defaultdict(list)
for k, (_, r, _) in enumerate(hidden):
    by_type[r].append(k)
lp.table([
    {"tipo de arista": r, "escondidas": len(ks), **{m: float(np.mean(1 / ranks_by_model[m][ks])) for m in MODELS}}
    for r, ks in sorted(by_type.items(), key=lambda item: -len(item[1]))
])

Y las quince primeras tripletas escondidas, con el rango de cada modelo:

In [ ]:
lp.table([{"h": h, "r": r, "t": t, **{m: int(ranks_by_model[m][k]) for m in MODELS}} for k, (h, r, t) in enumerate(hidden[:15])])

Cómo leerlas: un 1 es un acierto a la primera; un número alto, que la respuesta quedó muy abajo
entre todas las entidades. Fíjate en qué tipos de arista (`r`) dan rangos bajos y en cuáles altos,
y en si el origen `h` tiene otras aristas que el modelo haya podido ver.

En esta familia una arista escondida falta dos veces: como respuesta y como camino de mensajes. Un
enunciado que pierde su única arista `is_part_of` pierde también el mensaje de su cláusula, y su
vector se construye con lo poco que le queda.

## 5. La pregunta que nos importa: ¿qué parte?

Se esconde la arista de parte de un enunciado y se le pregunta al modelo cuál de las **dos**
partes encaja mejor. Con dos candidatas, el MRR no aporta nada que no diga el acierto (vale
`(1 + acierto) / 2`), así que se mide el **acierto**: la fracción de enunciados con la parte bien.

- Solo los **enunciados unilaterales** (la celda dice cuántos). Los gemelos —la misma frase
  emitida una vez por parte, como en *«each party shall…»*— tienen las dos partes por
  construcción y no hay nada que adivinar.
- Se reparten los unilaterales en cinco grupos; se esconde un grupo, se entrena con el resto del
  grafo y se pregunta por ese grupo. Así hasta pasar por los cinco, y todo `REPEATS` veces con
  repartos distintos.
- Dos referencias sin modelo: **siempre la parte más frecuente** y **la mayoría de su cláusula**
  (la parte que más aparece entre los otros enunciados de la misma cláusula).
- La parte «correcta» es la que puso la extracción: en este contrato nadie la ha revisado a mano.

In [ ]:
queries = [q for q in lp.party_queries(contract) if q[0] not in contract.twin]
twins = [q for q in lp.party_queries(contract) if q[0] in contract.twin]
print(f"{len(queries)} enunciados unilaterales, {len(twins)} gemelos")
parties = sorted(contract.parties)
correct = collections.defaultdict(list)  # método -> [acierto de cada repetición]
predictions = collections.defaultdict(dict)  # método -> {enunciado: parte} de la primera repetición
start = time.time()
for rep in range(REPEATS):
    fold_of = lp.folds(len(queries), 5, seed=rep)
    hits = collections.Counter()
    for fold in range(5):
        test = [q for q, f in zip(queries, fold_of) if f == fold]
        known = [q for q, f in zip(queries, fold_of) if f != fold] + twins
        graph = [t for t in contract.triples if t not in set(test)]
        base = lp.t1_baselines(contract, known, test)
        guesses = {"siempre la más frecuente": base["mayoría"], "mayoría de su cláusula": base["cláusula"]}
        for m in MODELS:
            scores = lp.kge_tail_scores(lp.train_kge(m, graph, e2i, r2i, seed=rep, **CONFIG[m]), [(h, r) for h, r, _ in test], e2i, r2i)
            guesses[m] = {sid: parties[int(np.argmax([row[e2i[p]] for p in parties]))] for (sid, _, _), row in zip(test, scores)}
        for method, guess in guesses.items():
            hits[method] += sum(guess[sid] == p for sid, _, p in test)
            if rep == 0:
                predictions[method].update(guess)
    for method, k in hits.items():
        correct[method].append(k / len(queries))
print(f"{REPEATS * 5 * len(MODELS)} entrenamientos en {time.time() - start:.0f} s")
lp.table(sorted([{"método": m, "acierto": float(np.mean(v)), "peor repetición": min(v), "mejor repetición": max(v)} for m, v in correct.items()], key=lambda r: -r["acierto"]))

En [`link_prediction.ipynb`](../link_prediction.ipynb), con estos mismos modelos y el mismo reparto,
R-GCN y CompGCN acertaron los dos el 55,3%, exactamente lo que da «siempre la más frecuente»; la
mayoría de su cláusula da el 61%, y la regla del sujeto (la primera parte que nombra la frase), el
76%. Esta tabla debería repetirlo.

### Por qué

Al esconder la arista de parte, ¿qué le queda al modelo para decidir? Las otras aristas del
enunciado, y lo que hay detrás de ellas. Cuántas aristas propias tiene cada uno de los
unilaterales:

In [ ]:
others = collections.Counter(
    sum(1 for h, r, t in contract.triples if sid in (h, t) and r not in lp.PARTY_RELATIONS) for sid, _, _ in queries
)
lp.table([{"aristas además de la de parte": k, "enunciados": others[k]} for k in sorted(others)])

Esas aristas son casi siempre `is_part_of` hacia su cláusula y, a veces, un término definido que
usa, una condición o un valor que cuelgan de él. **Ninguna dice quién carga con el enunciado.**

Con dos capas, el vector de un enunciado resume su cláusula y, a través de ella, los otros
enunciados de la cláusula; el de cada parte resume los enunciados que la apuntan. Al puntuar, el
modelo compara los dos: lo más que puede descubrir es que los vecinos del enunciado se parecen a
los de una parte, que es otra forma de decir la mayoría de su cláusula. Los que no tienen ninguna
otra arista quedan aislados: no les llega ningún mensaje.

Lo que dice quién carga está **en el texto**, no en el grafo. Unos casos en que R-GCN falla:

In [ ]:
fails = [(sid, p) for sid, _, p in queries if predictions["RGCN"][sid] != p][:6]
lp.table([{
    "enunciado": sid,
    "texto": lp.statement_text(contract.statements[sid])[:80],
    "lo que ve el modelo": ", ".join(f"{r} {t if h == sid else h}" for h, r, t in contract.triples if sid in (h, t) and r not in lp.PARTY_RELATIONS) or "nada",
    "RGCN dice": contract.parties[predictions["RGCN"][sid]]["name"][:22],
    "correcta": contract.parties[p]["name"][:22],
} for sid, p in fails])

## En resumen

- El grafo de Ediets son las tripletas `(h, r, t)` del paso 1.
- El paso 4 dice cuánto aprenden los dos del grafo en general, y en qué tipos de arista.
- El paso 5 dice si adivinan la parte de un enunciado mejor que mirar su cláusula. Si no lo hacen,
  la razón es la del apartado *Por qué*: las aristas que le quedan a un enunciado no dicen quién
  carga con él, y pasar mensajes por el grafo solo reparte lo que el grafo ya tiene.
- La configuración se ajustó en SteelVault y aquí se usa tal cual.